In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler 단계

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
이 페이지에서는 Qiskit SDK의 사전 구축된 트랜스파일레이션 파이프라인 단계를 설명합니다. 6개의 단계가 있습니다:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 함수는 이러한 단계로 구성된 사전 설정된 [단계별 패스 매니저](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager)를 생성합니다. 각 단계를 구성하는 특정 패스는 `generate_preset_pass_manager`에 전달되는 인수에 따라 달라집니다. `optimization_level`은 반드시 지정해야 하는 위치 인수이며, 0, 1, 2, 3 중 하나의 정수입니다. 값이 높을수록 더 강력하지만 비용이 많이 드는 최적화를 나타냅니다([트랜스파일레이션 기본값 및 구성 옵션](defaults-and-configuration-options) 참조).

Circuit을 트랜스파일하는 권장 방법은 사전 설정된 단계별 패스 매니저를 생성한 후 해당 패스 매니저를 Circuit에서 실행하는 것입니다([패스 매니저로 트랜스파일](transpile-with-pass-managers) 참조). 그러나 더 간단하지만 사용자 정의 가능성이 적은 대안으로 [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) 함수를 사용할 수 있습니다. 이 함수는 Circuit을 직접 인수로 받습니다. `generate_preset_pass_manager`와 마찬가지로 사용되는 특정 Transpiler 패스는 `transpile`에 전달되는 `optimization_level`과 같은 인수에 따라 달라집니다. 사실 내부적으로 `transpile` 함수는 `generate_preset_pass_manager`를 호출하여 사전 설정된 단계별 패스 매니저를 생성하고 Circuit에서 실행합니다.
## Init 단계
이 첫 번째 단계는 기본적으로 거의 아무것도 하지 않으며, 주로 자신만의 초기 최적화를 포함하려는 경우에 유용합니다. 대부분의 레이아웃 및 라우팅 알고리즘은 1-Qubit 및 2-Qubit Gate에서만 작동하도록 설계되어 있으므로, 이 단계는 2개 이상의 Qubit에서 작동하는 Gate를 1개 또는 2개의 Qubit에서만 작동하는 Gate로 변환하는 데에도 사용됩니다.

이 단계에 대한 자체 초기 최적화 구현에 대한 자세한 내용은 플러그인 및 패스 매니저 사용자 정의 섹션을 참조하세요.
## Layout 단계
다음 단계는 Circuit이 전송될 Backend의 레이아웃 또는 연결성과 관련됩니다. 일반적으로 양자 Circuit은 계산에 사용되는 실제 Qubit의 "가상" 또는 "논리적" 표현인 Qubit를 가진 추상 엔티티입니다. Gate 시퀀스를 실행하려면 "가상" Qubit에서 실제 양자 장치의 "물리적" Qubit으로의 일대일 매핑이 필요합니다. 이 매핑은 `Layout` 객체에 저장되며 Backend의 [명령어 세트 아키텍처(ISA)](./transpile#instruction-set-architecture) 내에 정의된 제약 조건의 일부입니다.

![이 이미지는 Qubit이 와이어 표현에서 QPU에서 Qubit가 연결된 방식을 나타내는 다이어그램으로 매핑되는 것을 보여줍니다.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Qubit 매핑")

매핑의 선택은 입력 Circuit을 장치 토폴로지에 매핑하는 데 필요한 SWAP 연산 수를 최소화하고 가장 잘 교정된 Qubit를 사용하는 데 매우 중요합니다. 이 단계의 중요성 때문에 사전 설정된 패스 매니저는 최적의 레이아웃을 찾기 위해 여러 방법을 시도합니다. 일반적으로 두 단계로 이루어집니다: 먼저 "완벽한" 레이아웃(SWAP 연산이 필요 없는 레이아웃)을 찾으려 시도하고, 완벽한 레이아웃을 찾을 수 없는 경우 최적의 레이아웃을 찾는 휴리스틱 패스를 시도합니다. 이 첫 번째 단계에서 일반적으로 사용되는 두 가지 `Passes`가 있습니다:

- `TrivialLayout`: 각 가상 Qubit를 장치의 동일 번호 물리적 Qubit에 단순하게 매핑합니다(예: [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). 이는 완벽한 레이아웃을 찾기 위해 `optimzation_level=1`에서만 사용되는 과거 동작입니다. 실패하면 다음으로 `VF2Layout`이 시도됩니다.
- `VF2Layout`: 이 단계를 VF2++ 알고리즘으로 해결되는 부분 그래프 동형 문제로 처리하여 이상적인 레이아웃을 선택하는 `AnalysisPass`입니다. 둘 이상의 레이아웃이 발견되면 평균 오류가 가장 낮은 매핑을 선택하기 위해 점수 휴리스틱이 실행됩니다.

그런 다음 휴리스틱 단계에서는 기본적으로 두 가지 패스가 사용됩니다:

- `DenseLayout`: 가장 높은 연결성을 가지고 Circuit과 동일한 수의 Qubit를 가진 장치의 하위 그래프를 찾습니다(Circuit에 제어 흐름 연산(예: IfElseOp)이 있는 경우 최적화 레벨 1에서 사용됨).
- `SabreLayout`: 초기 랜덤 레이아웃에서 시작하여 `SabreSwap` 알고리즘을 반복적으로 실행하여 레이아웃을 선택하는 패스입니다. `VF2Layout` 패스를 통해 완벽한 레이아웃을 찾지 못한 경우 최적화 레벨 1, 2, 3에서만 사용됩니다. 이 알고리즘에 대한 자세한 내용은 논문 [arXiv:1809.02573](https://arxiv.org/abs/1809.02573)을 참조하세요.
## Routing 단계
양자 장치에서 직접 연결되지 않은 Qubit 사이에 2-Qubit Gate를 구현하려면, 장치 Gate 맵에서 인접하게 될 때까지 Qubit 상태를 이동시키기 위해 하나 이상의 SWAP Gate를 Circuit에 삽입해야 합니다. 각 SWAP Gate는 비용이 많이 들고 노이즈가 있는 연산입니다. 따라서 Circuit을 주어진 장치에 매핑하는 데 필요한 최소 SWAP Gate 수를 찾는 것은 트랜스파일레이션 프로세스의 중요한 단계입니다. 효율성을 위해 이 단계는 기본적으로 Layout 단계와 함께 계산되지만, 논리적으로는 서로 구별됩니다. *Layout* 단계는 사용할 하드웨어 Qubit를 선택하고, *Routing* 단계는 선택된 레이아웃을 사용하여 Circuit을 실행하기 위해 적절한 양의 SWAP Gate를 삽입합니다.

그러나 최적의 SWAP 매핑을 찾는 것은 어렵습니다. 사실 이것은 NP-hard 문제이며, 따라서 가장 작은 양자 장치와 입력 Circuit을 제외하고는 계산 비용이 엄청나게 높습니다. 이를 해결하기 위해 Qiskit은 `SabreSwap`이라는 확률적 휴리스틱 알고리즘을 사용하여 좋지만 반드시 최적은 아닌 SWAP 매핑을 계산합니다. 확률적 방법의 사용은 반복 실행에서 생성된 Circuit이 동일하다는 것을 보장하지 않음을 의미합니다. 실제로 동일한 Circuit을 반복적으로 실행하면 출력에서 Circuit 깊이와 Gate 수의 분포가 생성됩니다. 이러한 이유로 많은 사용자가 라우팅 함수(또는 전체 `StagedPassManager`)를 여러 번 실행하고 출력 분포에서 가장 낮은 깊이의 Circuit을 선택합니다.

예를 들어, “나쁜”(연결되지 않은) `initial_layout`을 사용하여 100번 실행된 15-Qubit GHZ Circuit을 살펴보겠습니다.

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeAuckland()

ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/358cfb50-02fc-48f2-a4ec-657837e0c304-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/bb3b8c1f-69fd-4e0c-9b78-9ee67f3361bb-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](./transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

보시다시피 이 Circuit은 연결성 그래프에서 매우 멀리 떨어진 Qubit 0과 14 사이에 2-Qubit Gate를 실행해야 합니다. 따라서 이 Circuit을 실행하려면 `SabreSwap` 패스를 사용하여 모든 2-Qubit Gate를 실행하기 위해 SWAP Gate를 삽입해야 합니다.

또한 `SabreSwap` 알고리즘은 이전 단계의 더 큰 `SabreLayout` 메서드와 다릅니다. 기본적으로 `SabreLayout`은 레이아웃과 라우팅을 모두 실행하고 변환된 Circuit을 반환합니다. 이는 패스의 [API 참조 페이지](../api/qiskit/qiskit.transpiler.passes.SabreLayout)에 명시된 몇 가지 특정 기술적 이유 때문입니다.
## Translation 단계
양자 Circuit을 작성할 때, 원하는 양자 Gate(유니터리 연산)와 Qubit 측정이나 리셋 명령과 같은 비-Gate 연산의 집합을 자유롭게 사용할 수 있습니다. 그러나 대부분의 양자 장치는 소수의 양자 Gate 및 비-Gate 연산만 기본적으로 지원합니다. 이러한 기본 Gate는 대상의 [ISA](./transpile#instruction-set-architecture) 정의의 일부이며, 사전 설정된 `PassManagers`의 이 단계에서는 Circuit에 지정된 Gate를 지정된 Backend의 기본 기저 Gate로 변환(또는 *언롤*)합니다. 이는 Backend에서 Circuit을 실행할 수 있게 하는 중요한 단계이지만, 일반적으로 깊이와 Gate 수의 증가로 이어집니다.

두 가지 특수한 경우가 특히 중요하며, 이 단계가 수행하는 작업을 설명하는 데 도움이 됩니다.

1. SWAP Gate가 대상 Backend의 기본 Gate가 아닌 경우, 3개의 CNOT Gate가 필요합니다:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2507de9c-1b94-4d56-b5a6-0b2bb1719a80-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

3개의 CNOT Gate의 곱으로서, SWAP은 노이즈가 있는 양자 장치에서 수행하기에 비용이 많이 드는 연산입니다. 그러나 이러한 연산은 일반적으로 많은 장치의 제한된 Gate 연결성에 Circuit을 임베딩하는 데 필요합니다. 따라서 Circuit에서 SWAP Gate의 수를 최소화하는 것이 트랜스파일레이션 프로세스의 주요 목표입니다.

2. Toffoli 또는 제어-제어-not Gate(`ccx`)는 3-Qubit Gate입니다. 기저 Gate 세트가 1-Qubit 및 2-Qubit Gate만 포함하므로 이 연산은 분해되어야 합니다. 그러나 비용이 꽤 높습니다:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = FakeWashingtonV2()

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4339deb5-4947-493b-8940-e68c91311631-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

양자 Circuit의 모든 Toffoli Gate에 대해 하드웨어는 최대 6개의 CNOT Gate와 소수의 1-Qubit Gate를 실행할 수 있습니다. 이 예제는 여러 Toffoli Gate를 사용하는 모든 알고리즘이 큰 깊이의 Circuit이 되어 노이즈의 영향을 상당히 받게 됨을 보여줍니다.
## Optimization 단계
이 단계는 양자 Circuit을 대상 장치의 기저 Gate 세트로 분해하는 것을 중심으로 하며, Layout 및 Routing 단계에서 증가된 깊이에 대응해야 합니다. 다행히 Gate를 결합하거나 제거하여 Circuit을 최적화하는 많은 루틴이 있습니다. 일부 경우에 이러한 방법은 너무 효과적이어서 하드웨어 토폴로지에 대한 레이아웃 및 라우팅 후에도 출력 Circuit이 입력보다 더 낮은 깊이를 가집니다. 다른 경우에는 할 수 있는 것이 많지 않으며, 노이즈가 있는 장치에서 계산을 수행하기 어려울 수 있습니다. 이 단계에서 다양한 최적화 레벨이 차이를 보이기 시작합니다.

- `optimization_level=1`의 경우, 이 단계는 1-Qubit Gate 체인을 결합하고 연속된 CNOT Gate를 취소하는 [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)과 [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation)을 준비합니다.
- `optimization_level=2`의 경우, 이 단계는 `CXCancellation` 대신 교환 관계를 활용하여 중복 Gate를 제거하는 [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) 패스를 사용합니다.
- `optimization_level=3`의 경우, 이 단계는 다음 패스를 준비합니다:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

또한 이 단계는 Circuit의 모든 명령이 대상 Backend에서 사용 가능한 기저 Gate로 구성되어 있는지 확인하기 위한 몇 가지 최종 검사도 실행합니다.

아래 GHZ 상태를 사용한 예제는 다양한 최적화 레벨 설정이 Circuit 깊이와 Gate 수에 미치는 영향을 보여줍니다.

> **Note:** 확률적 SWAP 매퍼로 인해 트랜스파일레이션 출력이 달라집니다. 따라서 아래 숫자는 코드를 실행할 때마다 변경될 수 있습니다.

![15-Qubit GHZ 상태](../docs/images/guides/transpiler-stages/transpiler-11.avif "트랜스파일레이션 전 15-Qubit GHZ 상태")

다음 코드는 15-Qubit GHZ 상태를 구성하고 결과 Circuit 깊이, Gate 수, 멀티-Qubit Gate 수 측면에서 트랜스파일레이션의 `optimization_levels`를 비교합니다.